In [ ]:
import pickle
import pandas as pd

In [ ]:
with open('model_hw.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [ ]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [ ]:
df = read_data('../datasets/yellow_tripdata_2023-03.parquet')

In [ ]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

In [ ]:
y_pred.std()

In [ ]:
year = df['tpep_pickup_datetime'].dt.year.iloc[0]
month = df['tpep_pickup_datetime'].dt.month.iloc[0]

df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype(str)

In [ ]:
df.head()

In [ ]:
df_result = df[['duration',	'ride_id']]

output_file = 'results.parquet'

df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)

In [ ]:
del df

In [ ]:
df_april = read_data('../datasets/yellow_tripdata_2023-04.parquet')

dicts = df_april[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

In [ ]:
y_pred.mean()

In [ ]:
df_may = read_data('../datasets/yellow_tripdata_2023-05.parquet')
with open('./web-service/model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)
dicts = df_may[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

In [ ]:
y_pred.mean()